# MOOSE inference (nb2): converted NIfTI → segmentations  —  model-specific

Runs `moosez` clinical-CT models on the GPU VM. Consumes the **Boundary-A** archive
`converted_nifti.tar.lz4` from nb1 and emits the **Boundary-B** archive
`segmentations.tar.lz4` with the canonical layout:
```
<SeriesInstanceUID>/<model>/segmentations/<multilabel>.nii.gz
<SeriesInstanceUID>/<model>/label_map.json      # {label_id: label_name}
```
nb3 (shared) joins each `label_name` against the model's SNOMED mapping to build
DICOM-SEG + pyradiomics + SR. This notebook is the *only* MOOSE-specific piece.

## Imports

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import traceback
from pathlib import Path

NOTEBOOK_START = time.time()
def _elapsed(s=None):
    return f"{time.time() - (s if s is not None else NOTEBOOK_START):.1f}s"
print(f"[T+{_elapsed()}] Imports complete")

## Parameters

In [ ]:
# Boundary-A archive produced by nb1 (local file on the same VM).
converted_nifti_path = "converted_nifti.tar.lz4"

# Short model identifier used in the Boundary-B layout (<uid>/<model>/...).
model_name = "moose"

# 'cuda' for GPU, 'cpu' for CPU-only (falls back to cpu if no GPU).
accelerator = "cuda"

# Optional GCS prefix (gs://bucket/prefix) for checkpoint/resume on preemption: each
# finished (series, model) output is saved there and a retried VM skips it. run_id (the
# Cromwell workflow id, from the WDL) namespaces the checkpoint. Empty = disabled.
checkpoint_gcs = ""
run_id = ""

# Model-specific knob injected via `papermill -f inference_params.yaml`.
moose_models = "clin_ct_organs,clin_ct_ribs,clin_ct_vertebrae"

# Wall-clock cap per (series, worker launch); a hung worker is killed and recorded as an
# error instead of burning the VM. 0 = no cap.
series_timeout_s = 1800

# Wall-clock cap per (series, worker launch); a hung worker is killed and recorded as an
# error instead of burning the VM. 0 = no cap.
series_timeout_s = 1800


## Extract Boundary-A archive

In [ ]:
models = [m.strip() for m in moose_models.split(',') if m.strip()]
NIFTI_DIR = Path('/tmp/converted_nifti')
SEG_DIR = Path('/tmp/segmentations')
for _d in (NIFTI_DIR, SEG_DIR):
    if _d.exists():
        shutil.rmtree(_d)
    _d.mkdir(parents=True, exist_ok=True)

subprocess.run(f'lz4 -d -c {converted_nifti_path} | tar -xf - -C {NIFTI_DIR.parent}',
               shell=True, check=True)
# nb1 packs the directory 'converted_nifti/<uid>/<uid>.nii.gz'; collapse if nested.
if (NIFTI_DIR / 'converted_nifti').is_dir():
    NIFTI_DIR = NIFTI_DIR / 'converted_nifti'
series_uids = sorted(d.name for d in NIFTI_DIR.iterdir() if d.is_dir())
print(f'MOOSE models : {models}')
print(f'Series       : {len(series_uids)}')

# ---- checkpoint / resume (segmentator_checkpoint.py is fetched next to the notebook by
#      the WDL; no-op when checkpoint_gcs is empty) ----
import os, sys
for _p in (os.getcwd(), str(Path.cwd())):
    if _p not in sys.path:
        sys.path.insert(0, _p)
try:
    from segmentator_checkpoint import Checkpointer
except ImportError:
    Checkpointer = None
if checkpoint_gcs and Checkpointer is None:
    raise RuntimeError('checkpoint_gcs is set but segmentator_checkpoint.py was not found '
                       'next to the notebook (the WDL fetches it from gitRepo/gitBranch)')
ckpt = Checkpointer(checkpoint_gcs, run_id) if Checkpointer else None
completed_seg = ckpt.restore_segs(SEG_DIR) if ckpt else set()
if completed_seg:
    print(f'[T+{_elapsed()}] {len(completed_seg)} (series, model) outputs restored from checkpoint')

## GPU availability check (fall back to CPU if none)

In [ ]:
usage_metrics = {'series': {}, 'gpu': []}
try:
    import torch
    print(f'PyTorch {torch.__version__}  CUDA available: {torch.cuda.is_available()}')
    if not torch.cuda.is_available() and accelerator == 'cuda':
        print('WARNING: accelerator=cuda but no GPU found; falling back to cpu')
        accelerator = 'cpu'
except Exception as exc:
    print(f'PyTorch unavailable: {exc}')
    if accelerator == 'cuda':
        accelerator = 'cpu'
print(f'[T+{_elapsed()}] Accelerator = {accelerator}')

## Run moosez → Boundary-B layout (one worker process per series)

`moosez` writes `<out>/<uid>/moosez-<model>-<ts>/segmentations/<multilabel>.nii.gz`.
Each produced multilabel volume is moved to the canonical
`<uid>/<model>/segmentations/` and its `organ_indices` (moosez's own, authoritative
`{label_id: label_name}`) is written next to it as `label_map.json`.

moosez runs in a **separate process per series**: a native crash inside moosez/nnU-Net
(observed on Terra for `clin_ct_body_composition` on a chest CT whose L3 is edge-clipped —
a kernel death no `try/except` can catch) then costs only the model in flight. The worker
appends a JSON line to a progress file after every model; if it dies, the parent records
the in-flight model as an error and relaunches the worker for the remaining models, so the
other models of that series and every other series in the batch still complete.


In [ ]:
WORKER = Path('/tmp/moose_worker.py')
WORKER.write_text(r"""
# Per-series moosez worker (written by nb2). Runs the requested models for ONE series in
# its own process so a native crash cannot take down the notebook kernel. After every
# model a JSON record is appended to the progress file, so the parent can tell exactly
# which models finished even if this process dies mid-list.
import json, os, shutil, signal, sys, time, traceback
from pathlib import Path


def main():
    nii, work, series_dir, accelerator, progress = sys.argv[1:6]
    models = sys.argv[6].split(',')
    from moosez import moose
    with open(progress, 'a') as prog:
        for model in models:
            t0 = time.time()
            rec = {'model': model}
            try:
                if os.environ.get('MOOSE_WORKER_FAULT_INJECT') == model:
                    os.kill(os.getpid(), signal.SIGSEGV)   # test hook: simulate a native crash
                seg_paths, model_objs = moose(nii, [model], work, accelerator)
                rec['seconds'] = round(time.time() - t0, 1)
                if not seg_paths or not model_objs:
                    rec['error'] = 'moose produced no output'
                else:
                    dest = Path(series_dir) / model / 'segmentations'
                    dest.mkdir(parents=True, exist_ok=True)
                    for sp in seg_paths:
                        shutil.copy(str(sp), str(dest / Path(sp).name))
                    label_map = {str(k): v for k, v in dict(model_objs[0].organ_indices).items()}
                    (Path(series_dir) / model / 'label_map.json').write_text(
                        json.dumps({'model': model, 'labels': label_map}, indent=2))
                    rec['labels'] = len(label_map)
            except Exception:
                rec['seconds'] = round(time.time() - t0, 1)
                rec['error'] = traceback.format_exc()
            prog.write(json.dumps(rec) + '\n')
            prog.flush()


# The __main__ guard is essential: moosez/nnU-Net use spawn-based multiprocessing, which
# re-imports this file in every child process.
if __name__ == '__main__':
    main()
""")


def _run_worker(nii, work, series_dir, todo):
    """Run the worker for `todo` models; return ({model: record}, returncode, note)."""
    progress = work / f'progress_{int(time.time() * 1000)}.jsonl'
    cmd = [sys.executable, str(WORKER), str(nii), str(work), str(series_dir), accelerator,
           str(progress), ','.join(todo)]
    note = ''
    try:
        rc = subprocess.run(cmd, timeout=series_timeout_s or None).returncode
    except subprocess.TimeoutExpired:
        rc, note = -1, f'timed out after {series_timeout_s}s'
    if rc < 0 and not note:
        note = f'killed by signal {-rc}'
    elif rc > 0:
        note = f'exit code {rc}'
    recs = {}
    if progress.exists():
        for line in progress.read_text().splitlines():
            if line.strip():
                r = json.loads(line)
                recs[r['model']] = r
    return recs, rc, note


moose_errors = []
for uid in series_uids:
    nii = NIFTI_DIR / uid / f'{uid}.nii.gz'
    if not nii.exists():
        cands = list((NIFTI_DIR / uid).glob('*.nii.gz'))
        if not cands:
            moose_errors.append(f'{uid}: no NIfTI found')
            continue
        nii = cands[0]
    work = Path('/tmp/moose_work') / uid
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True, exist_ok=True)
    print(f'[T+{_elapsed()}] Series {uid}', flush=True)
    series_times = {}
    restored_models = [m for m in models if (uid, m) in completed_seg]
    todo = [m for m in models if m not in restored_models]
    series_dir = SEG_DIR / uid
    crashes = 0
    while todo:
        recs, rc, note = _run_worker(nii, work, series_dir, todo)
        for model in list(todo):
            rec = recs.get(model)
            if rec is None:
                break                       # this model was in flight when the worker died
            todo.remove(model)
            if rec.get('error'):
                moose_errors.append(f'{uid}/{model}: {rec["error"]}')
                print(f'  ERROR {model}: {rec["error"].strip().splitlines()[-1]}', flush=True)
                continue
            series_times[model] = rec['seconds']
            print(f'  {model}: {rec["seconds"]}s  ({rec["labels"]} labels)', flush=True)
            if ckpt:
                ckpt.save_seg(SEG_DIR, uid, model)
        if todo and (rc != 0 or not recs.get(todo[0])):
            # worker died (or timed out) with todo[0] in flight: record it, drop it, and
            # relaunch for whatever is left so the rest of the series still completes
            crashed = todo.pop(0)
            crashes += 1
            moose_errors.append(f'{uid}/{crashed}: moosez worker died ({note or "no output"})')
            print(f'  CRASH {crashed}: worker {note or "produced no record"}; '
                  f'{len(todo)} model(s) left, relaunching' if todo else
                  f'  CRASH {crashed}: worker {note or "produced no record"}', flush=True)
    # Propagate the exact input NIfTI (identical geometry to the masks) to the
    # Boundary-B series root so nb3 runs radiomics against it directly instead of
    # a second, independent dcm2niix conversion of the reference DICOM. Only when
    # at least one sub-model produced output (so the series dir exists).
    if series_dir.is_dir():
        shutil.copy(str(nii), str(series_dir / 'reference.nii.gz'))
    usage_metrics['series'][uid] = {'moose_models_s': series_times,
                                    'checkpoint_restored_models': restored_models,
                                    'worker_crashes': crashes}
    if restored_models:
        print(f'  restored from checkpoint: {restored_models}')

if moose_errors:
    Path('inference_errors.txt').write_text('\n'.join(moose_errors))
print(f'[T+{_elapsed()}] Inference complete ({len(moose_errors)} error(s))')


## Package Boundary-B archive + usage metrics

In [ ]:
import csv
import importlib.util
import urllib.request

produced = [d for d in SEG_DIR.iterdir() if d.is_dir()]
if not produced:
    raise RuntimeError('No segmentations produced — see inference_errors.txt')

# Bundle moosez's own SNOMED mapping (moose_snomed_mapping.csv) into the archive
# root so nb3 keys segments off the package's authoritative table rather than a
# hand-copied, drift-prone CSV. Prefer the file shipped inside the installed
# moosez package; fall back to fetching it from ENHANCE-PET/MOOSE for
# standalone / Colab runs where the package layout differs.
def _bundle_moose_snomed(dest):
    src = None
    spec = importlib.util.find_spec('moosez')
    for base in (spec.submodule_search_locations or []) if spec else []:
        cand = Path(base) / 'mappings' / 'moose_snomed_mapping.csv'
        if cand.exists():
            src = cand
            break
    if src is not None:
        shutil.copy(str(src), str(dest))
        print(f'Bundled SNOMED mapping from moosez package: {src}')
    else:
        url = ('https://raw.githubusercontent.com/ENHANCE-PET/MOOSE/'
               'main/moosez/mappings/moose_snomed_mapping.csv')
        urllib.request.urlretrieve(url, str(dest))
        print(f'moosez package mapping not found; fetched from {url}')

_bundle_moose_snomed(SEG_DIR / 'snomed_mapping.csv')

subprocess.run(f'tar -cf - -C {SEG_DIR.parent} {SEG_DIR.name} | lz4 > segmentations.tar.lz4',
               shell=True, check=True)
size_mb = Path('segmentations.tar.lz4').stat().st_size / (1024 ** 2)

usage_metrics['total_elapsed_s'] = round(time.time() - NOTEBOOK_START, 1)
with open('inference_UsageMetrics.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['SeriesInstanceUID', 'model', 'model_inference_s', 'run_total_elapsed_s',
                'checkpoint_restored'])
    for uid, m in usage_metrics['series'].items():
        for model, secs in m.get('moose_models_s', {}).items():
            w.writerow([uid, model, secs, usage_metrics['total_elapsed_s'], False])
        for model in m.get('checkpoint_restored_models', []):
            w.writerow([uid, model, '', usage_metrics['total_elapsed_s'], True])

# Inference finished and the archive is written: the checkpoint is no longer needed.
if ckpt:
    ckpt.cleanup()

print(f'[T+{_elapsed()}] Wrote segmentations.tar.lz4 ({size_mb:.1f} MB, {len(produced)} series)')